In [1]:
from pyiceberg.catalog import load_catalog
import duckdb

PROJECT = "/Users/Shared/academy/learner/projects/ethereum_iceberg"
catalog = load_catalog(
    "local",
    type="sql",
    uri=f"sqlite:///{PROJECT}/catalog/pyiceberg_catalog.db",
    warehouse=f"file://{PROJECT}/warehouse",
)
S3 = "s3://aws-public-blockchain/v1.0/eth"
con = duckdb.connect()
con.execute("set s3_region = 'us-east-2'")
catalog.list_namespaces()

[('ref',), ('raw',)]

In [2]:
from pyiceberg.transforms import DayTransform
first = con.execute(f"select * exclude (input) from read_parquet('{S3}/transactions/date=2026-10-01/*.parquet')").to_arrow_table()
transactions = catalog.create_table("raw.transactions", schema=first.schema)
with transactions.update_spec() as spec:
    spec.add_field("block_timestamp", DayTransform(), "block_day")
transactions.spec()

PartitionSpec(PartitionField(source_id=19, field_id=1000, transform=DayTransform(), name='block_day'), spec_id=1)

In [3]:
for day in range(1, 8):
    batch = first if day == 1 else con.execute(
        f"select * exclude (input) from read_parquet('{S3}/transactions/date=2026-10-0{day}/*.parquet')").to_arrow_table()
    transactions.append(batch)
    print(day, batch.num_rows)

1 2036041
2 1875646
3 1562292
4 1526358
5 1774270
6 1957602
7 1963594


In [4]:
transactions.inspect.partitions().select(["partition", "record_count", "file_count"]).to_pandas()

,partition,record_count,file_count
0,{'block_day': 2026-10-07},1963594,2
1,{'block_day': 2026-10-06},1957602,2
2,{'block_day': 2026-10-05},1774270,2
3,{'block_day': 2026-10-04},1526358,2
4,{'block_day': 2026-10-03},1562292,2
5,{'block_day': 2026-10-02},1875646,2
6,{'block_day': 2026-10-01},2036041,2


In [5]:
tasks = transactions.scan(row_filter="block_timestamp >= '2026-10-03T00:00:00' and block_timestamp < '2026-10-04T00:00:00'").plan_files()
[t.file.file_path.split("/data/")[1] for t in tasks]

['block_day=2026-10-03/00000-0-3c4ceb6b-1b03-45ab-b031-04709790a0a5.parquet',
 'block_day=2026-10-03/00000-1-3c4ceb6b-1b03-45ab-b031-04709790a0a5.parquet']

In [6]:
blocks = catalog.load_table("raw.blocks")
with blocks.update_spec() as spec:
    spec.add_field("timestamp", DayTransform(), "block_day")
nov3 = con.execute(f"""
    select * exclude (logs_bloom, last_modified, sha3_uncles, miner), miner as fee_recipient
    from read_parquet('{S3}/blocks/date=2026-11-03/*.parquet')
""").to_arrow_table()
blocks.append(nov3)

In [7]:
blocks.inspect.files().select(["spec_id", "partition", "record_count"]).to_pandas().sort_values("spec_id").tail(3)

,spec_id,partition,record_count
6,0,{'block_day': None},7173
5,0,{'block_day': None},215179
0,1,{'block_day': 2026-11-03},7182
